### Data Fetching

In [23]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_25732\1965381518.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7760 rows from 'extraction'


In [24]:
keywords = ["Signalling"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

len(df)

212

In [25]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['notification', 'signalling', 'work_order']


In [26]:
# import re
# import numpy as np
# import pandas as pd
# from collections import Counter

# na_like_values = ['NA', 'NULL', 'NONE', 'NAN']
# pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

# def is_na_value(value):
#     """Check if a value is considered 'NA' based on the pattern."""
#     if pd.isna(value) or value is None:
#         return True
#     if isinstance(value, str):
#         return bool(pattern_na.match(value))
#     return False

# def clean_value(value):
#     """Recursively converts 'NA' string values in dicts/lists to np.nan."""
#     if isinstance(value, dict):
#         return {k: clean_value(v) for k, v in value.items()}
#     elif isinstance(value, list):
#         return [clean_value(v) for v in value]
#     elif isinstance(value, str) and is_na_value(value):
#         return np.nan
#     else:
#         return value

# def find_na_keys(d, parent=''):
#     """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
#     na_keys = []
#     if isinstance(d, dict):
#         for k, v in d.items():
#             full_key = f"{parent}.{k}" if parent else k
#             if isinstance(v, dict):
#                 na_keys.extend(find_na_keys(v, full_key))
#             elif is_na_value(v): # Checks for string 'NA', None, and np.nan
#                 na_keys.append(full_key)
#     return na_keys

# def flattened_json(d):
#     flat_data = {}

#     def _flatten(data, parent_key=''):
#         if isinstance(data, dict):
#             for k, v in data.items():
#                 new_key = f"{parent_key}.{k}" if parent_key else k
                
#                 if isinstance(v, dict):
#                     _flatten(v, new_key)
#                 else:
#                     flat_data[new_key] = v
#         elif isinstance(data, list):
#             for i, item in enumerate(data):
#                 _flatten(item, f"{parent_key}[{i}]")

#     _flatten(d)
#     return flat_data

In [27]:
df_signalling = df.copy()

df_signalling['signalling_and_interlocking'] = df_signalling['json_data'].apply(
    lambda x: x.get('signalling', {}).get('signalling_and_interlocking') if isinstance(x, dict) else None
)

df_signalling['wayside_signalling'] = df_signalling['json_data'].apply(
    lambda x: x.get('signalling', {}).get('wayside_signalling') if isinstance(x, dict) else None
)

df_signalling['ctc'] = df_signalling['json_data'].apply(
    lambda x: x.get('signalling', {}).get('ctc') if isinstance(x, dict) else None
)

df_signalling.drop(columns=['json_data'], inplace=True)

In [28]:
df_ctc = df_signalling[df_signalling['ctc'].notna()].copy()
df_ctc.drop(columns=['signalling_and_interlocking', 'wayside_signalling'], inplace=True)

df_no_ctc = df_signalling[df_signalling['ctc'].isna()].copy()
df_no_ctc.drop(columns=['ctc'], inplace=True)

In [29]:
output_file = f"../../output/snc/signalling_system.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_no_ctc.to_excel(writer, sheet_name='station', index=False)
    df_ctc.to_excel(writer, sheet_name='occ_dcc', index=False)

print(f"Saved excel to {output_file}")

Saved excel to ../../output/snc/signalling_system.xlsx
